# AIM:
create evaluation workflow. Taking the manually extracted Ci impacts (validation set) and compare it with the CI impacts (llm_geolocations.ipynb) extrracted by the first LLM 1. 
As a first step the evaluation should be done only for the direct CI impacts - CI type, damage and geolocation

Issue:
* What is needed an approach that recognizes when an direct impact case is not detected by the model
Idea: 
* Split the original texts passed to the model on the exact chunks as again
* Then chunkwise check if the CI impacts from the validation set correspond in number and their textual similarity to the CI impacts infered by the LLM 1 and Entity Linking 

## Semantic Textual Similarity (STS)

Calculating the STS for both model configurations (chain of prompts, orchestration of models)
The outputs are cosine similarity scores for similar model outputs per chunk. They are ranked by score for each model, restricted to the top 20 results.  


In [1]:
import os
import sys
from pathlib import Path
import io
import gc
import time
import warnings
import subprocess
import importlib

from unidecode import unidecode
import langdetect
from fuzzywuzzy import fuzz
import torch
from huggingface_hub import login
import numpy as np
import spacy
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from matplotlib import pyplot as plt


sys.path.append('../')
from src.settings import settings as s
import src.document_cleaning as dc
import src.translation_model as tm
import src.utils as u
import src.datahandler as dh
import src.postprocess as pp

# login to HF
# NOTE raises exception when env.variable does not exist (compared to os.envrion.get and its shortcut os.getenv)
os.getenv("HUGGINGFACE_TOKEN")

#  automatic linebreaks and multi-line cells.
pd.set_option("display.colheader_justify", "left")
pd.set_option('display.max_colwidth', 5000)


/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


Running on local machine


### Direct CI impacts: LLM 1 vs domain-expertise 

In [ ]:
#  Suppress future warnings from PyTorch
warnings.filterwarnings("ignore", category=FutureWarning)


#  Define data dir where tags.csv and domain-expertise derived tag lists are found 
VALID_DATA_FILENAME = s.VALID_DATA_FILENAME
PATH_VALID_DATA = s.PATH_VALID_DATA
PATH_EVAL_RESULT = s.PATH_EVAL_RESULT
# s.LLM_DATA_FILENAME = "llm1_NER_newWF.csv"
s.LLM_DATA_FILENAME = "llm_1_updprompt_dNER.csv"
LLM_DATA_FILEPATH = Path(s.PATH_LLM_DATA / s.LLM_DATA_FILENAME)
SIMILARITY_LLM_FILENAME = s.SIMILARITY_LLM_FILENAME

df_valid_org = pd.read_csv(
    PATH_VALID_DATA / VALID_DATA_FILENAME,
    usecols=["publication_id", "ci1_type", "ci1_damage", "ci1_location", "sentence_reference"]
)
print(len(df_valid_org))
## pre-process: 
# remove undone entries
df_valid_org = df_valid_org[~df_valid_org.astype(str).apply(lambda x: x.str.contains("xx")).any(axis=1)]
# remove further location info (e.g. that entry is a town, Landkreis, Bavaria etc.)
df_valid_org["ci1_location"] = df_valid_org["ci1_location"].replace(r"\s*\(.*\)", "", regex=True).str.strip()
df_valid_org = df_valid_org.dropna(subset=["publication_id"], how="all") # drop rows where citation info is missing
print(len(df_valid_org))


## prediction data
df_pred = pd.read_csv(
    LLM_DATA_FILEPATH,
    #usecols=["citation_id", "chunk_id", "infrastructure_type", "damage", "location", "chunk_text"]
)

## citation alignment
# print(df_pred["citation_id"])
# df_pred["citation_id"] = df_pred["citation_id"].map(dc.extract_citation_info) # FIXME as used with new funct returning author, year, title
# df_pred["citation_id"] = df_pred["citation_id"].apply(dc.extract_citation_info)
# print(df_pred["citation_id"])


141
126


In [3]:
df_pred["citation_id"].unique()

array(['Karakatsani 2023', "Lloyd's List 2024", 'Containerlift 2024',
       'ABC 2024', 'Koks 2022', 'European Investment Bank 2025',
       'Wilson 2024', 'Wildhagen 2013', 'EFE 2024', 'Ferlita 2023',
       'Gilbody Dickerson 2024'], dtype=object)

#### AS FUNC: Evaluate on same documents that were passed to LLM



In [4]:
#  TODO make as global var

PARSED_TEXT_DIR = Path(s.PATH_DATA / "parsed_documents/")

docs_list_sample = [
        # Path(PARSED_TEXT_DIR, "AEMET 2024 - ESTUDIO SOBRE LA SITUACIÓN DE LLUVIAS INTENSAS_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Karakatsani 2023 - Greece economy briefing The economic impact of the recent devastating floods in Greece_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Lloyd's List 2024 - Port of Valencia reopens after devastating floods_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Containerlift 2024 - Valencia Port Resumes Operations Following Devastating Flooding in Spain - Containerlift.co.uk - Transport_Lifting_Shipping_cleaned.md"), 
            Path(PARSED_TEXT_DIR, "ABC 2024 - Traffic jams and flight delays due to heavy rain and lightning storm in Malaga_cleaned.md"),

            Path(PARSED_TEXT_DIR, "Koks 2022 - Brief communication_cleaned.md"),
        Path(PARSED_TEXT_DIR, "European Investment Bank 2025 - Spain_ EIB lends €50 million to Iberdrola to rebuild and climate-proof flood-hit power infrastructure in Valencia_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Wilson 2024 - Flash floods in Spain sweep away cars, disrupt trains and leave several missing _ AP News_cleaned.md"),     
            Path(PARSED_TEXT_DIR, "Wildhagen 2013 - Hochwasser_ Wie die Flut Unternehmen lahmlegt_cleaned.md"),

        # # not Deidda et al, IPCC, Fekete 2025 as it already contains coarse info about many CI impacts
            # Path(PARSED_TEXT_DIR, "AFP 2022 - The_Vibes_Valencia Airport in Madrid briefly shut as lightning hits runway _ World _ The Vibes_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Artemis 2015 - PERILS finalises Storm Desmond UK flood loss estimate at £604m_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Brown 2010 - Economy feels chill as UK grinds to a halt _ The Independent_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Diakakis 2020 - A systematic assessment of the effects of extreme flash floods on transportation infrastructure and circulation: The example of the 2017 Mandra flood_cleaned.md"),
        Path(PARSED_TEXT_DIR, "EFE 2024 - The DANA storm, live_ The death toll rises to 158_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Eurelectric 2006 - Impacts of Severe Storms on Electric Grids_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Euronews 2024 - Spain floods_ Death toll rises to 205 as nation braces for more rain _cleaned.md"),
        Path(PARSED_TEXT_DIR, "Ferlita 2023 - Incendi in Sicilia, ecco cosa accade_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Fink 2004 - The 2003 European summer heatwaves and drought - synoptic diagnosis and impacts_cleaned.md"),
        Path(PARSED_TEXT_DIR, "Gilbody Dickerson 2024 - Spain floods_ At least 95 people killed including British man near Malaga _ World News _ Sky News_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Kadir 2014 - The Impact of Natural Disasters on Critical Infrastructures - A Domino Effect-based Study_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Kaur 2025 - Authorities suspect arson in 17 wildfires across Dalmatian coast, Croatia - The Watchers_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Keller 2014 - Mapping Natural Hazard Impacts on Road Infrastructure—The Extreme Precipitation in Baden-Württemberg_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Kettle 2020 - Storm Xaver over Europe in December 2013 Overview of energy impacts and North Sea events_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Koks 2019 - Understanding Business Disruption and Economic Losses Due to Electricity Failures and Flooding_cleaned.md"),
            # Path(PARSED_TEXT_DIR, "Korzilius 2021 Nach der Flut_cleaned.md"),

        # Path(PARSED_TEXT_DIR, "Khazai 2013 - Juni-Hochwasser 2013 in Mitteleuropa - Fokus Deutschland Bericht 2 Auswirkungen und Bewältigung_cleaned.md"),
            
        # not part of valid set:
        # Path(PARSED_TEXT_DIR, "Krausmann 2014 - STREST report on lessons learned from recent catastrophic events_cleaned.md"), # > 1800 entries LLMv3.0 incl. hallucinations
]

In [5]:
citation_list = []

for i in docs_list_sample:
    a, y, t = dc.extract_citation_info(i.name)
    citation_list.append(a + y)

df_valid = df_valid_org[df_valid_org["publication_id"].isin(citation_list)]
df_valid.publication_id.unique()

array(['ABC 2024', 'Containerlift 2024', 'EFE 2024',
       'European Investment Bank 2025', 'Ferlita 2023',
       'Gilbody Dickerson 2024', 'Karakatsani 2023', 'Koks 2022',
       'Wildhagen 2013', 'Wilson 2024'], dtype=object)

In [6]:
df_pred[~df_pred["ci_entity"].isna()]


,citation_id,chunk_id,infrastructure_type,damage,location,ci_entity,geo_entity,case_type,chunk_text
0,Karakatsani 2023,0,roads,severely damaged,Thessaly plain,National roads,plain,NaN,"The economic impact of the recent devastating floods in Greece The economic impact of the recent devastating floods in Greece The briefing presents the economic impact of the damages caused by the storm “Daniel”. The floods caused by the heavy rainfall, especially in Thessaly plain -accounting for approximately 15% of Greece’s agricultural land- resulted to a massive destruction in agriculture, infrastructure and residences, which is expected to negatively affect the Greek economy in short and mid-term. Fears for food shortages and increase of product prices have been raised. In addition, increase of fiscal costs, imports and unemployment may also affect the economy in the upcoming years. The extent of the impact to the economy is not yet know. Before the country recovered from the wildfires of August and the consequent huge forest disaster, a weather stormy phenomenon named Daniel hit the country. Specifically, in the beginning of September, rainfall of unprecedented intensity fell mainly in central Greece and especially in Thessaly plain, causing floods throughout the territory. Thousands of acres of crops and livestock farms were destroyed. Properties and houses were lost under ton of waters. National roads were closed, bridges collapsed, and some parts of the railway network were highly damaged dividing the country in two. Evidently, the massive destruction of agricultural"
44,Karakatsani 2023,3,railway,partially destroyed,Thessaly plain,National highway,Thessaly plain,NaN,"water. Thus, the heartland of Greek agriculture is in a great extend destroyed and experts warn that the destruction of crops and soil quality will take years to recover. It is worth noting that a quarter of the country’s wheat and barley, 30 percent of cotton, a third of chickpeas, pistachios and lentils, a fifth of the hay used in livestock farming and half of the industrial tomato production was grown in Thessaly plain. Thus, it is expected that the massive destruction may result in shortages as well as increase in product prices (5). Furthermore, the storm resulted to the partial destruction of main roads, such as the National highway. The railway network has also been destroyed in some parts. According to the Minister of Infrastructure and Transportation, Christos Staikouras, the cost for repairing the damages in the railways will exceed 150 mil euros (6). The financial cost and the impact to the Greek economy From the above it is evident that the financial cost of repairing the damages brought by the storm “Daniel” is extremely high. According to government sources initially the cost of the damages in Thessaly was estimated approximately around 1.5 bil euros. However, new"
45,Karakatsani 2023,3,highway,heavily damaged,Thessaly plain,The railway network,Thessaly plain,NaN,"water. Thus, the heartland of Greek agriculture is in a great extend destroyed and experts warn that the destruction of crops and soil quality will take years to recover. It is worth noting that a quarter of the country’s wheat and barley, 30 percent of cotton, a third of chickpeas, pistachios and lentils, a fifth of the hay used in livestock farming and half of the industrial tomato production was grown in Thessaly plain. Thus, it is expected that the massive destruction may result in shortages as well as increase in product prices (5). Furthermore, the storm resulted to the partial destruction of main roads, such as the National highway. The railway network has also been destroyed in some parts. According to the Minister of Infrastructure and Transportation, Christos Staikouras, the cost for repairing the damages in the railways will exceed 150 mil euros (6). The financial cost and the impact to the Greek economy From the above it is evident that the financial cost of repairing the damages brough

In [7]:
df_pred.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1635 entries, 0 to 1634
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   citation_id          1635 non-null   object 
 1   chunk_id             1635 non-null   int64  
 2   infrastructure_type  1635 non-null   object 
 3   damage               1635 non-null   object 
 4   location             1635 non-null   object 
 5   ci_entity            84 non-null     object 
 6   geo_entity           84 non-null     object 
 7   case_type            0 non-null      float64
 8   chunk_text           1635 non-null   object 
dtypes: float64(1), int64(1), object(7)
memory usage: 115.1+ KB


In [8]:
df_pred.loc[df_pred["citation_id"]== "Krausmann 2014"] # Krausmann -> large hallucinations when chunk-text is title or contact info (i.e when not about CI /impacts)

,citation_id,chunk_id,infrastructure_type,damage,location,ci_entity,geo_entity,case_type,chunk_text


### drop dublicated cases which differ only in Tier 2 or Tier 3 impacts

> e.g. valid ABC 2024: - identical c1_type, ci1_damage, ci1_loc (but diff. ci2_damages -which are not used in this eval) 


In [9]:

print(f"Dropping {df_valid.duplicated().sum()} duplicates in valid data")
df_valid = df_valid.drop_duplicates()

print(f"Dropping {df_pred.duplicated().sum()} duplicates in pred data")
df_pred = df_pred.drop_duplicates()


Dropping 1 duplicates in valid data
Dropping 65 duplicates in pred data


In [10]:
 # df_pred.sort_values(["citation_id", "chunk_id", "infrastructure_type"]).loc[df_pred.duplicated(keep=False, subset=["citation_id", "chunk_id", "infrastructure_type", "damage", "location", "ci_entity", "geo_entity", "case_type", "chunk_text"])][50:]

### add unique identifiers
helps in calculating FPs and FNs

In [11]:
df_pred["id_pred"] = df_pred.reset_index().index
df_valid["id_valid"] = df_valid.reset_index().index

#### As FUNC. postprocess -make CI gsubgroups

### Improve similarity calculation
As all similarity measures - no matter which embedding model or kind of cosine similarity measure were not sufficient eg. port ~ power to similar to port~harbor

Thus, it might be better to first group ci impacts into subgroups e.g .based on HARCI-EU categories,as some kind of postprocessing step before applying the similarity measurements



In [12]:
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)


## group Ci types into subgroups,
df_pred = pp.group_ci_types(df_pred, "infrastructure_type", "infrastructure_group", ci_patterns)
df_valid = pp.group_ci_types(df_valid, "ci1_type", "ci1_group", ci_patterns)
## keep only records which are actually about CI (e.g., not theatre, stadion ..)
df_pred.dropna(subset=["infrastructure_group"], inplace=True)
df_valid.dropna(subset=["ci1_group"], inplace=True)

print(df_pred.infrastructure_group.isna().sum())  # mostly cases which are not CI (theater, stadion..)
print(df_pred.infrastructure_group.value_counts())
# df_pred.infrastructure_group.unique()



0
infrastructure_group
ports                           185
road_others                     143
it_telecommunication            128
rail                            118
healthcare_others               116
airports                        101
transport_others                 57
bridges                          55
education_kita                   52
education_others                 46
water_supply                     26
aviation                         24
healthcare_hospitals_clinics     21
motorways                        16
water_others                     15
education_school                 14
wastewater                        9
electricity_others                8
power_plants                      8
drinking_water                    6
waste_others                      6
railway_station                   3
waterprotection                   2
gas_distribution                  1
Name: count, dtype: int64


In [13]:
print(df_valid.ci1_group.isna().sum())
print(df_valid.ci1_group.value_counts())
# df_pred.infrastructure_group.unique()

0
ci1_group
road_others                     20
rail                             8
electricity_others               6
bridges                          4
healthcare_hospitals_clinics     4
airports                         3
wastewater                       3
motorways                        3
ports                            2
gas_distribution                 2
healthcare_others                2
water_others                     2
it_telecommunication             2
drinking_water                   2
waste_others                     1
water_supply                     1
education_kita                   1
education_school                 1
Name: count, dtype: int64


In [14]:
print("Removing all records which have another or erroneous CI entry")

df_pred = df_pred[~df_pred.infrastructure_group.isna()]
df_valid = df_valid[~df_valid.ci1_group.isna()]


Removing all records which have another or erroneous CI entry


#### Load spaCy language model


In [15]:
## load english model with contextual vectors included


## RELOAD spacy pipeline
nlp = spacy.load("./spacy_model_pipeline")

# add CI_TYPE patterns to spacy nlp model pipeline
config = {"spans_key": None, "annotate_ents": True, "overwrite": False}
## see for more info: https://spacy.io/usage/rule-based-matching#entityruler
## NOTE EntityRuler is hidden inside .add_pipe()
try:
    ruler = nlp.add_pipe("span_ruler", config=config)
    ruler.from_disk(s.NER_PATTERNS_FILEPATH)
except ValueError:
    print("SpanRuler already exists in pipeline.")
    ruler = nlp.get_pipe("span_ruler")
    ruler.from_disk(s.NER_PATTERNS_FILEPATH)


# # NOTE: en_core_web_lg can only return word vectors, while en_core_web_trf return contextual vectors (as transformer-based)
# !uv run python -m spacy download en_core_web_lg
# nlp = spacy.load("en_core_web_lg")




In [16]:
# ## unify citation column

# # get corresponding document from df_vald
# citation_pattern = r"(.*?)(\d{4})(.*)" # split at first occurrence of year
# # df_pred["publication_id"] = df_pred["citation"].map(dc.extract_citation_info)
# df_pred["citation"].map(dc.extract_citation_info)
# # df_pred = df_pred.rename({"citation": "publiation_id"})
# # df_pred.drop("citation", inplace=True)
# df_pred
# # try:
# #     authors, year, _ = re.findall(citation_pattern, filename)[0]
# #     citation = f"{authors} {year}"


#### Select records which have text references

In [17]:
df_valid_org.info()

<class 'pandas.core.frame.DataFrame'>
Index: 126 entries, 0 to 125
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   publication_id      126 non-null    object
 1   sentence_reference  126 non-null    object
 2   ci1_type            102 non-null    object
 3   ci1_damage          96 non-null     object
 4   ci1_location        87 non-null     object
dtypes: object(5)
memory usage: 5.9+ KB


In [18]:

print(len(df_pred), len(df_valid))
df_pred = df_pred[~df_pred["chunk_text"].isna()].reset_index(drop=True)
df_valid = df_valid[~df_valid["sentence_reference"].isna()].reset_index(drop=True)
print(len(df_pred), len(df_valid))


1160 67
1160 67


#### Translation of validation sentences

In [19]:

for entry in df_valid.itertuples():
    
    src_language = langdetect.detect(str(entry.sentence_reference))
    
    if src_language != "en":
        supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
        if src_language not in supported_languages:
            print(f"Unsupported source language: {src_language}. Continue with original version of the sentence in validation set ")
            continue 

        print(f"\n ######## -------- Translating {entry.publication_id}: {src_language} --> en -------- ######## \n")

        # # clean up before applying translator
        # gc.collect()
        # torch.cuda.empty_cache()  # mainly after training needed, small effect when LLM applied only for inference
        # torch.no_grad()
        
        # overwrite original sentence(s) with translated versions
        translated_sentence = tm.translate_2_english(src_language, str(entry.sentence_reference))
        df_valid.loc[df_valid.index[df_valid["sentence_reference"] == entry.sentence_reference], "sentence_reference"] = translated_sentence



 ######## -------- Translating Ferlita 2023: it --> en -------- ######## 

/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub
Using device: cuda
Using locally saved model from /home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/huggingface_mirror/hub
Input document is a string (not DoclingObject). Wrapping it in a list for processing.
Continue with translation of text string


#### Postprocess (text cleaning)


In [20]:
df_valid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   publication_id      67 non-null     object
 1   sentence_reference  67 non-null     object
 2   ci1_type            67 non-null     object
 3   ci1_damage          61 non-null     object
 4   ci1_location        54 non-null     object
 5   id_valid            67 non-null     int64 
 6   ci1_group           67 non-null     object
dtypes: int64(1), object(6)
memory usage: 3.8+ KB


In [21]:
# unicode to ascii representation

for col in ["infrastructure_group", "infrastructure_type", "damage", "ci_entity", "geo_entity"]:
    df_pred[col] = df_pred[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan


for col in ["ci1_group", "ci1_type", "ci1_damage", "ci1_location"]:
    df_valid[col] = df_valid[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan


#### Merge prediction entries with potential validation entries (nth:1 pairs)

In [22]:
print("Match chunk text of each prediction entry with related validation entries (nth:1 pairs)")

df_pred_valid_all = pd.DataFrame()
threshold = 75

# find for each prediction entry all validation entries for respective chunk 
# these validation entries are candidates from which the most similar one to the pred. entry is taken to calc. model performance 
# including also entries where pred_info or valid_info is missing (e.g FNs, FPs)
for _, pred_entry in df_pred.iterrows():
    for _, valid_entry in df_valid.iterrows():  # all validation entries of all docs

        if valid_entry.sentence_reference is np.nan:
            continue

        # Calculate match score by accounting for partial string matches. 
        # In detail, it calculates the similarity ratio using the shortest string (length n, here: "sentence_reference") against all n-length substrings of the larger string and returns the highest score 
        score = fuzz.partial_ratio(valid_entry['sentence_reference'], pred_entry['chunk_text'])

        if score >= threshold:
            entry_pred_valid = {
                "citation_id": pred_entry["citation_id"],
                "ci_pred": pred_entry["infrastructure_type"],
                "ci_group_pred": pred_entry["infrastructure_group"],
                "damage_pred": pred_entry["damage"],
                "location_pred": pred_entry["location"],
                "chunk_id_pred": pred_entry["chunk_id"],
                "chunk_text_pred": pred_entry["chunk_text"],
                "ci_valid": valid_entry["ci1_type"],
                "ci_group_valid": valid_entry["ci1_group"],
                "damage_valid": valid_entry["ci1_damage"],
                "location_valid": valid_entry["ci1_location"],
                "sentence_text_valid": valid_entry["sentence_reference"],
                "text_similarity": score,
                "id_pred": pred_entry["id_pred"],
                "id_valid": valid_entry["id_valid"]
            }
            df_pred_valid_all = pd.concat([df_pred_valid_all, pd.DataFrame([entry_pred_valid])], ignore_index=True)  # n:1 relationship DF
        

# 85 threshold - 778 entries
# 75 threshold - 778 entries
# 75 threshold + CIsubgrou - 513 entries




Match chunk text of each prediction entry with related validation entries (nth:1 pairs)


In [23]:
df_pred_valid_all.info() # 143 -190 entries

## --> FPs are more common compared to FNs, especially for predicting locations, 
# as it is easier to get a prep-valid match when pred.info is actually missing due to larger chunk-text (pred set) compared to sentence-text (valid set)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 554 entries, 0 to 553
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   citation_id          554 non-null    object
 1   ci_pred              554 non-null    object
 2   ci_group_pred        554 non-null    object
 3   damage_pred          554 non-null    object
 4   location_pred        554 non-null    object
 5   chunk_id_pred        554 non-null    int64 
 6   chunk_text_pred      554 non-null    object
 7   ci_valid             554 non-null    object
 8   ci_group_valid       554 non-null    object
 9   damage_valid         479 non-null    object
 10  location_valid       466 non-null    object
 11  sentence_text_valid  554 non-null    object
 12  text_similarity      554 non-null    int64 
 13  id_pred              554 non-null    int64 
 14  id_valid             554 non-null    int64 
dtypes: int64(4), object(11)
memory usage: 65.1+ KB


In [24]:
## entries with lowest similarity
df_pred_valid_all.text_similarity.describe()
# df_pred_valid_all.iloc[df_pred_valid_all.text_similarity.sort_values(ascending=True).index] [["sentence_text_valid", "chunk_text_pred","text_similarity"]]

count    554.000000
mean      99.592058
std        1.570412
min       87.000000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: text_similarity, dtype: float64

## Calc similarities 
* TPs (for all cases where text info in pred and valid set exists)
* FNs  (model missed actual cases)
* FPs  (model hallucinated cases)

In [ ]:
list_entity_valid = ["ci_group_valid", "damage_valid", "location_valid"]
list_entity_pred = ["ci_group_pred", "damage_pred", "location_pred"]



#  Set similarity threshold (self-defined) when CI case is valid or not FN/FP
cos_smlrty_thresh = 0.7
pr_smlrty_thresh = 80


print(" --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---")
print("Using 100% match for CI types based on subgroups")
print("Using cosine similarity threshold for damages", cos_smlrty_thresh)
print("Using cosine similarity threshold for damages", cos_smlrty_thresh)
print("Using partial ratio similarity threshold for locations", pr_smlrty_thresh)

## AIM of evaluation loop below: 
# remove all cases in df_pred_valid_all where pred_entities were wrongly assigned to a valid_entity
## ie keep only pre-valid pairs with highest similarity per unique valid case



df_eval_records = pd.DataFrame()


# For each impact case (rows) 
for record_no, impact_record in df_pred_valid_all.iterrows():

    print(f"Record: {record_no } / {len(df_pred_valid_all)}")

    # init dict to store results for each records (row=)
    df_eval = {
        "citation": impact_record.citation_id,
        "chunk_text_pred": impact_record.chunk_text_pred,
        "sentence_text_valid": impact_record.sentence_text_valid,
        "id_pred": impact_record.id_pred,
        "id_valid": impact_record.id_valid
    }
    
    # iterate over the three entity classes (ci, damage, location) to assess LLM performance
    for entity_valid, entity_pred in zip(list_entity_valid, list_entity_pred):


        # Calculate similarities for entries in column pair: entity_pred - entity_valid

        ## calc similarity when both pred_info and valid_info exist (ie. not NaN)
        if impact_record[entity_pred] and impact_record[entity_valid] is not np.nan:
            pred_impact = impact_record[entity_pred]
            valid_impact = impact_record[entity_valid]

            if entity_pred == "ci_group_pred": # for CI group, only partial ratio similarity is calculated as it is more important to get the correct group than the exact match (e.g. "port infrastructure" <-> "port")

                embedded_list = u.vector_calculation(pred_impact, valid_impact)
                # similarity_score_cos = u.cosine_similarity(embedded_list[0], embedded_list[1])
                # similarity_score_pr = np.nan
                
                # similarity on idential match 
                if pred_impact == valid_impact:
                    ci_smlrty = 1
                else:
                    ci_smlrty = 0

                # store result for ci entity 
                df_eval["ci_pred"] = pred_impact  # CI subgroup
                df_eval["ci_valid"] = valid_impact # CI subgroup
                df_eval["ci_smlrty"] = ci_smlrty

            if entity_pred == "damage_pred": 
                ## Cosine similarity calc.
                # contextual vectors (transformer-based)
                embedded_list = u.vector_calculation(pred_impact, valid_impact)
                # calculate cosine similarity for each pred-valid pair
                similarity_score_cos = u.cosine_similarity(embedded_list[0], embedded_list[1])  # 0-1 value, the higher the more similar

                # store result for DAM and LOC entity 
                df_eval["dam_pred"] = pred_impact  
                df_eval["dam_valid"] = valid_impact 
                df_eval["dam_smlrty"] = similarity_score_cos


            if entity_pred == "location_pred": 
                ## Cosine similarity calc.
                # contextual vectors (transformer-based)
                embedded_list = u.vector_calculation(pred_impact, valid_impact)
                # calculate cosine similarity for each pred-valid pair
                similarity_score_cos = u.cosine_similarity(embedded_list[0], embedded_list[1])  # 0-1 value, the higher the more similar
                ## Partial ratio similarity calc. (especially for locations and CI-type  "port infrastructure" <-> "port")
                similarity_score_pr = fuzz.partial_ratio(pred_impact, valid_impact)  

                # store result for DAM and LOC entity 
                df_eval["loc_pred"] = pred_impact  
                df_eval["loc_valid"] = valid_impact 
                df_eval["loc_smlrty"] = similarity_score_cos
                df_eval["loc_smlrty_pr"] = similarity_score_pr

    # collect all single records (row) with similarity scores
    df_eval_records = pd.concat([df_eval_records, pd.DataFrame([df_eval])], ignore_index=True)


## NOTE Description: How 1:1 pairs for pred-valid are extracted f
## 1. group by single records from df_valid (via id_valid indices), 
##    Column "id_valid": index represents single records from df_valid (when validation_sentence contains 2 cases: -> id-valid:0, id_valid:1,  sentence w 1 case: id-valid:2) 
## 2. then collect from each group the one with highest similarity to predictions
##    --> binary "mask" indicates where we have matches -e.g. correct predictions (true: TP, false: FN or FP)  is our match (1:1 pred-valid pair) - from which TPs can be calculated

## 1. + 2.
# select for each single valid record (ie rows in df_valid) the 1:1 match (pred-valid pair, "head(1)") with highest similarities across all three classes 
# NOTE need to sort based on all three smlrty cols to do correct Tp calc 
#      (if sort_values by on similartiy column would result in too many TPs- as then 1:1 pairs would contain also random matches where randomly CI_red is identical with CI_valid)
df_smltry_selmax = df_eval_records.groupby("id_valid").apply(lambda s: s.sort_values(["ci_smlrty","dam_smlrty","loc_smlrty", "loc_smlrty_pr"], ascending=False).head(1))
# # OLD  (makes too many 1:1 pairs as described in NOTE)
# mask = df_eval_records.groupby("id_valid").apply(lambda x: x==x["ci_smlrty"].max()).droplevel(0)
# df_smltry_selmax2 = df_eval_records.where(mask.ci_smlrty==mask.ci_smlrty.max()).dropna(how="all") # keep cases only which have highest similarity scores
# df_smltry_selmax2.reset_index(drop=True, inplace=True)


print("for each unique valid record keep only pred-valid pairs of highest similarity")


# iterate over the three entity classes (ci, damage, location) to assess LLM performance
for _, column_pred in zip(list_entity_valid, list_entity_pred):

    if column_pred == "ci_group_pred":

        # remove cases where no CI could be found (for Ci unlikelky, but more common for location or damage)
        df_valid_ci = df_valid[df_valid["ci1_group"].notnull()]
        df_pred_ci = df_pred[df_pred["infrastructure_group"].notnull()]

        # when no similarity could be calculated
        # ## FIXME move outside of loop
        # entries_with_no_similarity = df_eval_records.loc[df_eval_records["impact_sim_identical"].isna()]
        # print(f" --- Pred-valid pairs where no identical similarity score could be calculated: {len(entries_with_no_similarity)} ----")
        # print(entries_with_no_similarity[["impact_valid", "impact_pred", "impact_sim_identical", "impact_sim_cos", "impact_sim_pr", "citation"]])

        
        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["ci_smlrty"] == 1]
        print(len(tps), len(df_valid_ci )) 
        
        
        # # FPs
        # --> make mask where records in df-eval record are identical to df_pred.columns (must be 1:1), 
        #     aplly mask on df_pred and substract from output all cases which are in TPs 
        # assert len(output) == fps_len
        
        # FNs
        ## missed docs
        df_valid_ci_pred_missed_docs = df_valid_ci[df_valid_ci["publication_id"].isin(df_pred["citation_id"]) == False]
        ## missed entries
        # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
        ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of corectly predicted CI cases (Tps)
        ## no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_group"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1])

        # ## TODO FIXME not sure if approach for df_valid_cases_missed_by_model based on df_smltry_selmax is correct
        # ##            as df_smltry_selmax contains only the cases of highest similarity for each case in df_valid (ie unique id_valid)
        # ##            can i then calc the number of missed cases by 
        
        # fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)

        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_ci["infrastructure_group"]) - tps.shape[0]
        fns_len = len(df_valid_ci["ci1_group"]) - tps.shape[0]

        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)
                    
        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps), fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0

        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")

        # saving
        dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)


    if column_pred == "damage_pred":

        # remove cases where no CI could be found (for CI unlikelky, but more common for location or damage)
        df_valid_dam = df_valid[df_valid["ci1_damage"].notnull()]
        df_pred_dam = df_pred[df_pred["damage"].notnull()]  # when model gave NaN (then actually also corresponding df_valid record would be there NaN)

        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["dam_smlrty"] >= cos_smlrty_thresh]

        # ## check if TPs calc correct
        # # tps_validmergedpred = df_valid.merge(df_pred, left_on=["ci1_damage"], right_on=["damage"], how="inner") ## ERROR as gives > 4000 entries
        # # assert len(tps) == len(tps_validmergedpred)

        # # FPs - model predicts condition wrongly (ie. predict condition when it is actually absent)
        # # get all valid. documents which were also used for LLM inference
        # df_valid_pred_same_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"])]
        # print(f"Doing evaluation based on {df_valid_pred_same_docs.publication_id.unique().__len__()} documents existing in both (valid.+pred. set)")
        # # get records where model predicted presence of impacts but they actually does not exist
        # fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] < cos_smlrty_thresh]
        # # here as definition, that when simi=0 (or below threshold) then model predicted presences as false alarm
        # # TODO
        # # add also as Fps were model_pred case exist but no fitting_valid case could be found (during df_valid_pred pair generation in loop at begin of NB)
        # # df_pred selction needed

        # # FNs - CI impact cases not detected by model 
        # # NOTE: maybe FNs number is biased as wrong matches more likely as chunk-text (pred set) is longer than sentence text (valid set)
        # # WRONG? get all entries from df_valid_pred_same_docs where corresponding pred_record (in FPs) is missing

        # # get all documents in valid_set which does not occur in pred_set or where similarity is too low
        # ## missed docs
        # df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
        # print("Number of documents where model did not extract anything", df_valid_pred_missed_docs.shape)
        # ## missed entries
        # # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        # df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
        # ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of correctly predicted CI cases (Tps)
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_damage"])  - len(df_smltry_selmax["impact_valid"])
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_location"])  - len(df_smltry_selmax["impact_valid"])
        # fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)


        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_dam["damage"]) - tps.shape[0]  # that
        fns_len = len(df_valid_dam["ci1_damage"]) - tps.shape[0]

        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)

        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps), fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0

        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")
        
        ## saving
        dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)
        

    if column_pred == "location_pred":

        # remove cases where no CI could be found (for CI unlikelky, but more common for location or damage)
        df_valid_loc = df_valid[df_valid["ci1_location"].notnull()]
        df_pred_loc = df_pred[df_pred["location"].notnull()]  # when model gave NaN (then actually also corresponding df_valid record would be there NaN)

        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["loc_smlrty_pr"] >= pr_smlrty_thresh]

        # ## check if TPs calc correct
        # # tps_validmergedpred = df_valid.merge(df_pred, left_on=["ci1_location"], right_on=["location"], how="inner") ## ERROR as gives > 4000 entries
        # # assert len(tps) == len(tps_validmergedpred)

        # # FPs - model predicts condition wrongly (ie. predict condition when it is actually absent)
        # # get all valid. documents which were also used for LLM inference
        # df_valid_pred_same_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"])]
        # print(f"Doing evaluation based on {df_valid_pred_same_docs.publication_id.unique().__len__()} documents existing in both (valid.+pred. set)")
        # # get records where model predicted presence of impacts but they actually does not exist
        # fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] < cos_smlrty_thresh]
        # # here as definition, that when simi=0 (or below threshold) then model predicted presences as false alarm
        # # TODO
        # # add also as Fps were model_pred case exist but no fitting_valid case could be found (during df_valid_pred pair generation in loop at begin of NB)
        # # df_pred selction needed

        # # FNs - CI impact cases not detected by model 
        # # NOTE: maybe FNs number is biased as wrong matches more likely as chunk-text (pred set) is longer than sentence text (valid set)
        # # WRONG? get all entries from df_valid_pred_same_docs where corresponding pred_record (in FPs) is missing

        # # get all documents in valid_set which does not occur in pred_set or where similarity is too low
        # ## missed docs
        # df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
        # print("Number of documents where model did not extract anything", df_valid_pred_missed_docs.shape)
        # ## missed entries
        # # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        # df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
        # ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of correctly predicted CI cases (Tps)
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_damage"])  - len(df_smltry_selmax["impact_valid"])
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_location"])  - len(df_smltry_selmax["impact_valid"])
        # fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)
        
        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_loc["location"]) - tps.shape[0]
        fns_len = len(df_valid_loc["ci1_location"]) - tps.shape[0]
        
        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)

        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps),fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0
        
        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")

        ## saving
        dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)




### OLD WF
# 20 37
# tps 20  fps: 190  fns: 17
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.5405405405405406, Precision: 0.09523809523809523, F1-score: 0.16194331983805665
# tps 13  fps: 197  fns: 19
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.40625, Precision: 0.06190476190476191, F1-score: 0.10743801652892562
# tps 9  fps: 201  fns: 20
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.3103448275862069, Precision: 0.04285714285714286, F1-score: 0.07531380753138076


# ### NEW WF (no geollm, LLM1+old prompt+NER)
# 21 37
# tps 21  fps: 148  fns: 16
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.5675675675675675, Precision: 0.1242603550295858, F1-score: 0.20388349514563106
# tps 13  fps: 156  fns: 19
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.40625, Precision: 0.07692307692307693, F1-score: 0.12935323383084577
# tps 10  fps: 159  fns: 19
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.3448275862068966, Precision: 0.05917159763313609, F1-score: 0.10101010101010101


# llm_1_updprompt_dNER.csv
# 45 67
# tps 45  fps: 1115  fns: 22
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.6716417910447762, Precision: 0.03879310344827586, F1-score: 0.07334963325183375
# tps 14  fps: 1146  fns: 47
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.22950819672131148, Precision: 0.01206896551724138, F1-score: 0.022932022932022934
# tps 11  fps: 1149  fns: 43
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.2037037037037037, Precision: 0.009482758620689655, F1-score: 0.018121911037891267


 --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---
Using 100% match for CI types based on subgroups
Using cosine similarity threshold for damages 0.7
Using cosine similarity threshold for damages 0.7
Using partial ratio similarity threshold for locations 80
Record: 0 / 554


Record: 1 / 554
Record: 2 / 554
Record: 3 / 554
Record: 4 / 554
Record: 5 / 554
Record: 6 / 554
Record: 7 / 554
Record: 8 / 554
Record: 9 / 554
Record: 10 / 554
Record: 11 / 554
Record: 12 / 554
Record: 13 / 554
Record: 14 / 554
Record: 15 / 554
Record: 16 / 554
Record: 17 / 554
Record: 18 / 554
Record: 19 / 554
Record: 20 / 554
Record: 21 / 554
Record: 22 / 554
Record: 23 / 554
Record: 24 / 554


In [ ]:
# old
# Recall: 0.7377049180327869, Precision: 0.8653846153846154, F1-score: 0.7964601769911505

# fixed partly recall (FNs)
# Recall: 0.6716417910447762, Precision: 0.8653846153846154, F1-score: 0.7563025210084034

# new LLM extraction with fixed NERpatterns
# Recall: 0.6716417910447762, Precision: 0.8653846153846154, F1-score: 0.7563025210084034


### First figures  

In [ ]:
## histogram plot showing the number of documents over the years




## histogram plot showing which are most common in the prediction set


## ## a map of the locations of the infrastrucutre damages 





#### FIXME: find out which cases model predicted existence, but not in valid DS - maybe due that valid DS is incomppete?

In [ ]:
df_pred.info()

In [ ]:
## fix FPs 

## get all pred cases which 
# rows in df_valid where sentence_reference appears as substring in at least one df_pred.chunk_text
chunk_texts = df_pred["chunk_text"].dropna().astype(str)

df_valid_2 = df_valid[
    df_valid["sentence_reference"].fillna("").astype(str).apply(
        lambda s: any(s and s in chunk for chunk in chunk_texts)
    )
]

df_valid_2 # .shape (28, 7)

In [ ]:
# cases where pred-case exist but no fitting valid case could be found based on sentence_reference
df_pred_not_in_valid = df_pred[df_pred['chunk_text'].str.contains('|'.join(df_valid["sentence_reference"]), regex=True)]
print(df_pred_not_in_valid.id_pred.value_counts())
df_pred_not_in_valid.head(10)

# TODO TODO
## documents where model found many CI cases as false-alarms:
# maybe i need to recheck those docs and make df_valid more complete
# print(df_pred_not_in_valid.groupby("citation_id").count())
# citation_id                                                            
# ABC 2024                        4
# Containerlift 2024             24
# European Investment Bank 2025  21
# Ferlita 2023                   22
# Koks 2022                      10


In [ ]:
df_valid

#### FIXME: FPs and FNs

In [ ]:
## TPs + FNs should be == len(df_valid.ci) == 67
        
# TPs 
tps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 1]

# FNs
df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)

print(tps.shape[0], fns.shape[0])
print(tps.shape[0] + fns.shape[0])

# --> 8 cases in FNs are too definitly too much --> fix FN calculation



## FPs should be == len(df_pred.ci) - TPs

## FPs
fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 0]

print(len(df_pred.infrastructure_type), tps.shape[0], fps.shape[0])
print(len(df_pred.infrastructure_type) - tps.shape[0])




In [ ]:
# TODO fix FPs
# get records where model predicted presence of impacts but they actually does not exist
# here as definition, that when simi=0 (or below threshold) then model predicted wrongly
df_smltry_not_sim = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 0]

# TODO
# add also as Fps were model_pred case exist but no fitting_vlaid case could be found

# idea: 
# get all df_pred cases where chunk text not occurs in valid.sentece_text

df_pred_not_in_valid = df_pred[df_pred["chunk_text"].isin(df_valid["sentence_reference"])== False]
print(df_pred.shape, df_pred_not_in_valid.shape)
# df_pred_not_in_valid

In [ ]:
# df_valid__pred_no_thresh.id_pred.nunique()
df_pred_valid_no_thresh.id_pred.nunique()

In [ ]:
# df_valid__pred_no_thresh.info()
# df_pred_valid_no_thresh.info()  # 67 valid * 921 pred = 61707
# df_pred_valid_no_thresh.drop("chunk_text_pred", axis=1).sort_values("id_pred").iloc[0:100]
# df_pred_valid_no_thresh.groupby("id_pred").first().sort_values("text_similarity", ascending=False).iloc[0:100]
# df_valid__pred_no_thresh.groupby("id_valid").first().sort_values("text_similarity", ascending=False).iloc[0:100]
#df_valid__pred_no_thresh.sort_values("id_valid", ascending=False).sort_values("text_similarity", ascending=False).iloc[0:100]
#df_valid__pred_no_thresh.groupby("id_valid").first().sort_values("text_similarity", ascending=False).iloc[0:100]
df_valid__pred_no_thresh.groupby("id_valid").apply(lambda x: x.loc[x["text_similarity"].idxmax()])


In [ ]:
# fns

In [ ]:
## FNs 
df_smltry_selmax.loc[df_smltry_selmax.duplicated("id_pred")]
## --> ISSUE: this df (cases of highest sim) should NOT have duplicated cases of predictions -> maybe have to group based on id_pred and not id_valid

## try to fix issue
## --> currently i think this should group based on valid cases to measure were model predicted the same or missed info (ie FNs)
# df_smltry_selmax_p = df_smltry_selmax
# mask of rows with highest similarity score for each set of preds with unique valid case (droplevel(0) remove multiindex)
mask = df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max()).droplevel(0)
df_smltry_selmax_p = df_smltry_all.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similairty score
# df_smltry_selmax_p = df_smltry_all.groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) # 52 cases
# df_smltry_selmax_p = df_smltry_all.groupby("id_pred").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) # 175 cases
df_smltry_selmax_p.reset_index(drop=True, inplace=True)

## FIXME  df_smltry_all.groupby("id_valid"): should it has duplicated cases of id_pred ? - i dont think so! 
#  bc it means that there model missed cases in valid_set
## --> so all duplicated cases (except one-this is TP or FP) are actual FNs
print(df_smltry_selmax_p.info())
print(df_smltry_selmax_p.id_pred.nunique()  )  # should be len of df
print(df_smltry_selmax_p.duplicated().sum())


# FNs: extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
fns = df_smltry_selmax_p[df_smltry_selmax_p.duplicated(subset="id_pred", keep="first")]

print(fns.info())
fns.id_pred.value_counts()


In [ ]:
# return all cases which has max sim also when max score is shared by multiple rows 
# df_smltry_all.loc[df_smltry_all.groupby("id_valid").transform(lambda x: x==x.max()).astype('bool')].shape
mask = df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max())
mask = mask.droplevel(0)
#.transform(lambda x: x==x.max())
tt = df_smltry_all.loc[df_smltry_all.id_valid==37]#
tt = tt.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similairty score

# tt[mask]

# would return only first case of max sim:
#df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) #


In [ ]:
# FPs. 
print("False alarms (where model predicted ci but no corresponding valid case exists)", 
      len(df_pred["infrastructure_type"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]== 1, "ci_group_pred"])
    )
# Get FPs - cases where model predicted presence of CI (but actually it is absent in valid set)
tt = df_pred.merge(
    # FIXME issue that df_pred_valid_all contains some duplicates where id_pred identical but not valid_entries
    df_smltry_selmax.drop_duplicates(), # safety: make sure that merging is done on 1:1 match
    left_on="id_pred",#["citation_id", "chunk_id","infrastructure_type", "damage", "location"], 
    right_on="id_pred",#["citation_id", "chunk_id_pred", "ci_pred", "damage_pred", "location_pred"],
    how="left",
    indicator=True    # return an extra column indicating which table the row was from.
)
tt = tt.loc[tt["_merge"] == "left_only"].drop(columns=["_merge"])
print("False positives (model predicted CI but no corresponding valid case exists):", len(tt))

In [ ]:
print(df_pred.shape[0])
# print(df_pred_valid_all.info())
print(tt.info())

In [ ]:
df_pred#["infrastructure_type"]

In [ ]:
df_smltry_selmax.info()

In [ ]:
# df_smltry_selmax["impact_sim_identical"] < cos_smlrty_thresh

In [ ]:
# len(df_valid_pred_same_docs["ci1_group"]) 

In [ ]:
# TODO fix FNs
print(df_valid_pred_same_docs.info())
print(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1].info())
# --> FNS should be  23
len(df_valid_pred_same_docs["ci1_group"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1]["impact_valid"])

In [ ]:
# #df_valid_pred_same_docs["id_valid"] = df_valid_pred_same_docs.apply(lambda x: f"{x['ci1_group']}_{x['ci1_damage']}_{x['ci1_location']}_{x['sentence_text_valid'][:50]}", axis=1)
# print(df_valid_pred_same_docs["id_valid"].unique().__len__())
# print(df_valid_pred_same_docs.shape[0])
# ## --> check why electricity_others_outages_nan = 3  - (seems correct as sentences_ref are diff). airports_affected_Malaga area=2 are not unique
# df_valid_pred_same_docs[df_valid_pred_same_docs["id_valid"] == "airports_affected_Malaga area"]

# Improve similarity calculation
As all similarity measures - nomatter which emebdding model or kind of cosine similarity measure) were not sufficient eg. port ~ power to similar to port~harbor

Thus, it might be better to first group ci impacts into subgroups e.g .based on HARCI-EU categories,as some kind of postprocessing step before applying the similarity measurements



In [ ]:
df_ner = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

In [ ]:
# d = {"label":"CI_TYPE","pattern": [{"TEXT": {"REGEX": ".* [Tt]ransport.*"}}, {"LOWER": "sector"}], "subgroup_name":"transport_others"}
# #d["pattern"][0]["TEXT"]["REGEX"] + " " + dd["pattern"][1]["LOWER"]
# d["subgroup_name"]

In [ ]:
import re

# load regular expressions and subgroups from NER patterns as dict
# for general cases and all special cases with "LOWER"-pattern
regexes_1 = [
        {df_ner["pattern"][i][0]["TEXT"]["REGEX"] : df_ner["subgroup_name"][i]}
          for i in range(len(df_ner)) 
            if len(df_ner["pattern"][i])==1 
]
regexes_2 = [
    {df_ner["pattern"][i][0]["TEXT"]["REGEX"] + " " + df_ner["pattern"][i][1]["LOWER"] : df_ner["subgroup_name"][i] }
      for i in range(len(df_ner))
        if len(df_ner["pattern"][i])==2
]
# regexes = regexes_1 | regexes_2
regexes = regexes_1 + regexes_2
regexes[:20]


In [ ]:
for i, r in enumerate(regexes):

    # get regex pattern for CI type (key) and its subgroup (value)
    # NOTE, nice shortcut: get key containing regex by unpacking each dict into list, then get key
    pattern = [*r][0]
    subgroup = r[pattern]

    # assign subgroups to CI records, na=False to remove all records which not match patterns
    mask = df_pred["infrastructure_type"].str.contains(pattern, regex=True, na=False)
    df_pred.loc[mask, "infrastructure_group"] = subgroup

    # assign subgroups to CI records, na=False to remove all records which not match patterns
    mask = df_valid["ci1_type"].str.contains(pattern, regex=True, na=False)
    df_valid.loc[mask, "ci1_group"] = subgroup
    
df_pred

print(df_pred.infrastructure_group.isna().sum())
print(df_pred.infrastructure_group.value_counts())
# df_pred.infrastructure_group.unique()


In [ ]:
print(df_valid.df_valid.isna().sum())
print(df_valid.df_valid.value_counts())
# df_pred.infrastructure_group.unique()

In [ ]:
# s = "dyke" to s2 = "levee", s3 = "dam"
# bge-m3: 0.48  0.54
# all-MIniLM-L6-v2: 0.34 , 0.36  (similar all-mpnet-base-v2)
# gensim word2vec: 0.39 0.40


# s1 = "aviation" s2 = "air traffic"
# word vector spacy: 0.45
# contextual vector spacy: 0.68
# bge-m3: 0.76
# all-MIniLM-L6-v2: xx  (all-mpnet-base-v2: 0.79)
# gensim word2vec: 


# s1 = "power" s2 = "electricity"
# word vector spacy: 0.61
# contextual vector spacy: 0.66
# bge-m3: 
# all-MIniLM-L6-v2: xx   (all-mpnet-base-v2: 0.43)
# gensim word2vec: 0.58


# s1 = "electricity infrastructure" s2 = "electricity"
# word vector spacy: 0.87
# contextual vector spacy: 0.71
# bge-m3: 
# all-MIniLM-L6-v2:   xx  (all-mpnet-base-v2: 0.63)
# gensim word2vec: 


# s1 = "transportation" s2 = "transport infrastructure"
# word vector spacy: 
# contextual vector spacy: 
# bge-m3: 
# all-MIniLM-L6-v2:   xx  (all-mpnet-base-v2: 0.84)
# gensim word2vec: 

# s1 = "port" s2 = "power"  s3= harbour
# bge-m3: 0.58, 0.50
# all-MIniLM-L6-v2: 0.33 , 0.56  (similar all-mpnet-base-v2)
# gensim word2vec: 0.14 0.59

# s1 = "electricity" s2 = "transportation" 
# bge-m3:  0.64
# all-MIniLM-L6-v2:   (all-mpnet-base-v2: 0.47)
# gensim word2vec: 0.33

In [ ]:
# # print(cos_sim(model_scs["transportation"], model_scs["transport infrastructure"]))
# # print(cos_sim(model_scs["electricity infrastructure"], model_scs["electricity"]))
# # print(cos_sim(model_scs["power plant"], model_scs["electricity"]))
# print(cos_sim(model_scs["power"], model_scs["electricity"]))
# print(cos_sim(model_scs["aviation"], model_scs["air traffic"]))
# # identical to model_scs.similarity("port", "power"))


# # similarity_score = 1-distance.cosine(model.encode([s1])[0], model.encode([s2])[0])

### Analyse evaluation results 


In [ ]:
df_smltry_selmax#.info()

In [ ]:
## find out for which docs model performed bad (or good)
## based on this info try to improve model 

df_smltry_selmax.dropna(subset=["impact_sim_cos"]).groupby("citation").apply(lambda x: x.loc[x["impact_sim_cos"].idxmax()]).sort_values(by="impact_sim_cos", ascending=True)
## check EFE, Wilson, European Investment Bank, Containerlift, Lloyds List, Gilbody Dickerson


In [ ]:
## check entries of worst performace docs for damage
df_smltry_selmax.loc[df_smltry_selmax["citation"].isin(["Khazai 2023", "ABC 2024", "Containerlift 2024", "Lloyds List 2024", "Ferlita 2023"])]

In [ ]:
## check entries of worst performance docs for Ci tyes
df_smltry_selmax.loc[df_smltry_selmax["citation"].isin(["EFE 2024", "Containerlift 2024", "Lloyds List 2024", "Wilson 2024", "Gilbody Dickerson 2024", "European Investment Bank 2025"])].head(50)


## For each validation entry, search for all prediction cases of the same chunk 

In [ ]:
## get same impact entries
list_entity_valid = ["ci1_type", "ci1_damage", "ci1_location"]
list_entity_pred = ["infrastructure_type", "damage", "location"]


for entity_valid, entity_pred in zip(list_entity_valid, list_entity_pred):

    print(f" --------- Processing column pair: {entity_valid} - {entity_pred} ------------")
    
    df_valid_pred_all = pd.DataFrame()
    citations_list = []

    ## for each validation record
    for i in range(len(df_valid)):
        
        highest_similarity_score = 0.00
        
        ## needed to traceback info when entry is missing in pred. DS
        # chunk_id_value_valid = df_valid.chunk_id[i]

        # select nth validation record and check that it has value
        df_valid_entry = df_valid.iloc[i]
        if df_valid_entry[entity_valid] is np.nan:
            continue
        
        citation_str = df_valid_entry.publication_id
        citations_list.append(citation_str)


        # get all corresponding prediction records
        df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]

        #  handle on NANs
        df_pred_entries[entity_pred] = np.where(df_pred_entries[entity_pred].isna(), "nan", df_pred_entries[entity_pred])
        # df_pred_entries[entity_pred] = df_pred_entries[entity_pred].astype(str)
        # remove double whitespaces
        # df_pred_doc[entity_pred] = df_pred_doc[entity_pred].replace("  ", " ")
        # df_valid_entries[entity_valid] = df_valid_entries[entity_valid].replace("  ", " ")


        # vector of validiation entry 
        valid_impact = df_valid_entry[entity_valid]
        valid_vec = nlp(valid_impact).vector

        # print(" ------- Searching for citation:", citation_str, " in predictions ------- ")

        # Compute similarity between each validation CI impact case and all potential predicted CI impact cases (cross-product)
        for j in range(len(df_pred_entries[entity_pred])):

            if df_pred_entries[entity_pred].iloc[j] == "nan":
                continue

            pred_impact = df_pred_entries[entity_pred].iloc[j]

            pred_vec = nlp(pred_impact).vector
            similarity_score = u.cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
            # print(f"Similarity {i}-{j}: {similarity_score}")

            # print(f"Searching for highest similarity ... ")
            ## get only pair with highest similarity
            if similarity_score > highest_similarity_score:
                
                highest_similarity_score = similarity_score
                
                dict_pair = {
                    "impact_valid": valid_impact, 
                    "impact_pred": pred_impact, 
                    "similarity": highest_similarity_score,
                    "citation": citation_str,
                    "chunk_id_pred": (df_pred.chunk_id[i],  df_pred.chunk_id[j])
                }
            else:
                continue

        df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


    print(f" ---------- Evaluation summary statistics - {entity_pred}: -----------")
    print(df_valid_pred_all.similarity.describe())

    

    SIMILARITY_FILENAME = f'{entity_pred}_{SIMILARITY_LLM_FILENAME}'
    SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    print("Saving evaluation statistics, distribution plots, and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
    with open(SIMILARITY_FILEPATH, 'w') as f:
        # results
        df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
        df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
        pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
        #   summary statistics
        df_valid_pred_all_stats = df_valid_pred_all.describe()
        f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
        df_valid_pred_all_stats.to_json(f, indent=4)
        # distribution plots
        df_valid_pred_all.similarity.hist(bins=100).to_file(SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_hist.png")



    # cos_smlrty_thresh = 0.75
    # df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] <= cos_smlrty_thresh
    # print(f"Number of similar impact cases (similarity >= {cos_smlrty_thresh}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}\n")

    # df_valid_pred_all =  df_valid_pred_all[df_valid_pred_all['similarity'] <= cos_smlrty_thresh]

    # SIMILARITY_FILENAME = f'{entity_pred}_lower75_{SIMILARITY_LLM_FILENAME}'
    # SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    # with open(SIMILARITY_FILEPATH, 'w') as f:
    #     # results
    #     df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
    #     df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
    #     pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
    #     #   summary statistics
    #     df_valid_pred_all_stats = df_valid_pred_all.describe()
    #     f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
    #     df_valid_pred_all_stats.to_json(f, indent=4)


In [ ]:
# df_valid_pred_all[df_valid_pred_all['similarity'] <= 0.75]

# df_valid_pred_all.similarity.hist(bins=100)

In [ ]:
list_entity_pred

In [ ]:
LLM_DATA_FILEPATH

### Load parquet file

In [ ]:

list_entity_pred = ["infrastructure_type", "damage", "location"]

In [ ]:
entity_pred = "infrastructure_type"
SIMILARITY_FILENAME = f'llm1_similarity_{entity_pred}_75.parquet'
SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    df = pd.read_parquet(SIMILARITY_FILEPATH, engine='pyarrow')
    display(df)

## Archive

In [ ]:
## Aim 
## for all identical valid entries ie. with same [ci_valid	damage_valid	location_valid	sentence_text_valid]
## get the match to pred_entity with highest similarity

In [ ]:
    # ## calc for each entry with the same chunk_text the similarity between valid_impact and pred_impact
    # ## means we calc also the False Negatives (ie. where valid entry exists but no prediction)


    # # iterate over groups of entities which refer to the same valid case (i.e. which are identical in valid_columns)
    # # TODO iterate over unqiue cases in df_valid (instead of using grouper)
    # grouper = df_pred_valid_all[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]].drop_duplicates()
    # for group in grouper.itertuples():
    #     df_pred_valid_group = df_pred_valid_all[df_pred_valid_all[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]] == group[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]]]

    #     # calc. similarities to pred_entities
    #     for i, entry in df_pred_valid_group.iterrows():

    #         highest_similarity_score = 0 

    #         if entry[entity_pred].iloc[i] == "nan":
    #             continue
            
    #         # calc embeddings
    #         pred_impact = entry[entity_pred].iloc[i]
    #         pred_vec = nlp(pred_impact).vector

    #         valid_impact = entry[entity_valid].iloc[i] # is unique for each group
    #         valid_vec = nlp(valid_impact).vector
    #         print(valid_impact, "valid_impact")
            
    #         similarity_score = u.cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
    #         # print(f"Similarity {i}-{j}: {similarity_score}")

    #         ## return only pred-valid-pair with highest similarity
    #         if similarity_score > highest_similarity_score:
                
    #             highest_similarity_score = similarity_score
                
    #             entry["impact_similarity"] = highest_similarity_score

    # ## FNs
    # # # calc FN when valid_info exists but not corresponding pred_info
    # ## number of FNs is small due that wrong matching with any chunk-text is more likely due to its text size comapred sentence-level (valid set) 
    # elif entry[entity_pred] is np.nan:
    #     similarity_score = 0
    #     dict_pair = {
    #         "impact_valid": valid_impact, 
    #         "impact_pred": pred_impact, 
    #         "impact_similarity": similarity_score,
    #         "tp_tn_fp_fn": "fn",
    #         "citation": entry.citation_id,
    #         "chunk_text_pred": entry.chunk_text_pred,
    #         "sentence_text_valid": entry.sentence_text_valid,
    #         }
    #     df_smltry_selmax = pd.concat([df_smltry_selmax, pd.DataFrame([dict_pair])], ignore_index=True)

    # ## FPs
    # elif entry[entity_valid] is np.nan:
    #     similarity_score = 0
    #     dict_pair = {
    #         "impact_valid": valid_impact, 
    #         "impact_pred": pred_impact, 
    #         "impact_similarity": similarity_score,
    #         "tp_tn_fp_fn": "fp",
    #         "citation": entry.citation_id,
    #         "chunk_text_pred": entry.chunk_text_pred,
    #         "sentence_text_valid": entry.sentence_text_valid,
    #         }
    #     df_smltry_selmax = pd.concat([df_smltry_selmax, pd.DataFrame([dict_pair])], ignore_index=True)




In [ ]:
# ## get same impact entries
# list_entity_valid = ["ci1_type", "ci1_damage", "ci1_location"]
# list_entity_pred = ["infrastructure_type", "damage", "location"]



## iterate over predictions and search for each prediction reocrds for corresponding valid cases 

# for entity_valid, entity_pred in zip(list_entity_valid, list_entity_pred):

#     print(f" --------- Processing column pair: {entity_valid} - {entity_pred} ------------")
    
#     df_valid_pred_all = pd.DataFrame()
#     citations_list = []

#     ## for each validation record
#     for i in range(len(df_valid)):
        
#         highest_similarity_score = 0.00
        
#         ## needed to traceback info when entry is missing in pred. DS
#         # chunk_id_value_valid = df_valid.chunk_id[i]

#         # select nth validation record
#         df_valid_entry = df_valid.iloc[i]
#         citation_str = df_valid_entry.publication_id
#         citations_list.append(citation_str)
#         print(" ------- Searching for citation:", citation_str, " in predictions ------- ")


#         # get all corresponding prediction records
#         df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]
#         #  handle on NANs
#         df_pred_entries[entity_pred] = np.where(df_pred_entries[entity_pred].isna(), "nan", df_pred_entries[entity_pred])
#         # df_pred_entries[entity_pred] = df_pred_entries[entity_pred].astype(str)
#         # remove double whitespaces
#         # df_pred_doc[entity_pred] = df_pred_doc[entity_pred].replace("  ", " ")
#         # df_valid_entries[entity_valid] = df_valid_entries[entity_valid].replace("  ", " ")

#         # skip when validation entry ha no value
#         if df_valid_entry[entity_valid] is np.nan:
#             continue

#         # vector of validiation entry 
#         valid_impact = df_valid_entry[entity_valid]
#         valid_vec = nlp(valid_impact).vector


#         # Compute similarity between each predicted impact case and all potential validation impact cases (cross-product)
#         # print(f"Searching for highest similarity of`{pred_impact}` in validation set ... ")
#         for j in range(len(df_pred_entries[entity_pred])):

#             if df_pred_entries[entity_pred].iloc[j] == "nan":
#                 continue

#             pred_impact = df_pred_entries[entity_pred].iloc[j]

#             pred_vec = nlp(pred_impact).vector
#             similarity_score = u.cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
#             # print(f"Similarity {i}-{j}: {similarity_score}")

#             ## get only pair with highest similarity
#             if similarity_score > highest_similarity_score:
                
#                 highest_similarity_score = similarity_score
                
#                 dict_pair = {
#                     "impact_valid": valid_impact, 
#                     "impact_pred": pred_impact, 
#                     "similarity": highest_similarity_score,
#                     "citation": citation_str,
#                     "chunk_id_pred": df_pred.chunk_id[i]
#                 }
#             else:
#                 continue

#         df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


#     print(" ---------- Evaluation summary statistics: -----------")
#     print(df_valid_pred_all.similarity.describe())



#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{entity_pred}.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     print("Saving evaluation statistics and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)



#     cos_smlrty_thresh = 0.75
#     df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] >= cos_smlrty_thresh
#     print(f"Number of similar impact cases (similarity >= {cos_smlrty_thresh}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}")

#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{entity_pred}_75.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)


In [ ]:

# #  Define folder for handling and writing outputs
# def write_to_file(data, out_folder, filename):
#     """Convert output to DataFrame and write to file"""
#     df = pd.DataFrame(list(data), columns=['tag', 'sts_score'])
#     #  Sort the DataFrame by similarity (explicitly)
#     df = df.sort_values(by='sts_score', ascending=False)
#     #  Assign integers to ranking
#     df['rank'] = df['sts_score'].rank(method='first', ascending=False).astype(int)
#     #  Only keep the first 20 resulting tags
#     df = df.head(50)
#     #  Save to file
#     df.to_csv(out_folder / f'{filename}_output.csv', index=False)

# #  Fill run metrics to dictionary
# def handle_metrics(metrics, model_name, length, end_time, start_time):
#     print(f'-> Took {end_time - start_time:.2f} seconds. Number of tags: {length}.')
#     metrics.append({
#         'modelname': model_name,
#         'runtime': round(end_time - start_time, 2),
#         'tagcount': length
#     })
#     return metrics

# class CPU_Unpickler(pickle.Unpickler):
#     """Fix for having issues with loading models on CPU"""
#     def find_class(self, module, name):
#         if module == 'torch.storage' and name == '_load_from_bytes':
#             return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
#         else: return super().find_class(module, name)
